In [ ]:
import numpy as np
import pandas as pd
from time import time

# 1. Data Loading & Preparation (Assume X, y are loaded)
# Load data from 'ACDC_radiomics.csv' with error handling
try:
    df = pd.read_csv('ACDC_radiomics.csv')
    X = df.drop('class', axis=1)

    # Convert 'class' column to numerical using Label Encoding
    from sklearn.preprocessing import LabelEncoder
    label_encoder = LabelEncoder()
    y = label_encoder.fit_transform(df['class'])

    n_classes = 5
    print(f"Data shape: {X.shape}, Target shape: {y.shape}, Number of classes: {n_classes}")

except FileNotFoundError:
    print("Error: 'ACDC_radiomics.csv' not found in the current directory.")

# --- Configuration ---
N_SPLITS = 5        # Number of folds in StratifiedKFold
N_ITER_SEARCH = 50   # Number of parameter combinations for RandomizedSearchCV (adjust based on time)
RANDOM_STATE = 42    # For reproducibility

# Choose primary metric to optimize for and report others
# Common choices: 'accuracy', 'f1_macro' (good for imbalance), 'roc_auc' (good overall)
REFIT_METRIC = 'accuracy'

# --- Core Components ---
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA # Option 1: Feature Extraction
from sklearn.feature_selection import SelectKBest, f_classif # Option 2: Feature Selection

# Models
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

# Metrics
from sklearn.metrics import make_scorer, accuracy_score, f1_score, roc_auc_score, precision_score, recall_score

# --- 1. Define Cross-Validation Strategy ---
cv_strategy = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

# --- 2. Define Scoring Metrics ---
scoring = {
    'accuracy': make_scorer(accuracy_score),
    'f1_macro': make_scorer(f1_score, average='macro', zero_division=0),
    'precision_macro': make_scorer(precision_score, average='macro', zero_division=0),
    'recall_macro': make_scorer(recall_score, average='macro', zero_division=0),
}

# --- 3. Define Model Pipelines & Hyperparameter Spaces ---

# Common step: Scaling is crucial for SVM and k-NN, and doesn't hurt RFC
scaler = StandardScaler()

# Common step: Dimensionality Reduction (Choose one or test both types)
# Option A: PCA
dim_reduction_pca = PCA(random_state=RANDOM_STATE)
# Option B: SelectKBest (using ANOVA F-value for classification)
dim_reduction_kbest = SelectKBest(score_func=f_classif)

# --- Pipeline Definitions (Example using PCA) ---
# You might want to create separate experiments using SelectKBest

pipe_rfc = Pipeline([
    ('scaler', scaler),
    ('reduce_dim', dim_reduction_pca), # Or dim_reduction_kbest
    ('classifier', RandomForestClassifier(random_state=RANDOM_STATE, class_weight='balanced')) # balanced for potential imbalance
])

pipe_svm = Pipeline([
    ('scaler', scaler),
    ('reduce_dim', dim_reduction_pca), # Or dim_reduction_kbest
    ('classifier', SVC(random_state=RANDOM_STATE, probability=True, class_weight='balanced')) # probability=True for roc_auc
])

pipe_knn = Pipeline([
    ('scaler', scaler),
    ('reduce_dim', dim_reduction_pca), # Or dim_reduction_kbest
    ('classifier', KNeighborsClassifier())
])

# --- Hyperparameter Grids (for RandomizedSearchCV) ---
# Use distributions for continuous params where appropriate if desired (e.g., loguniform for C)

# Example using PCA - adjust n_components range based on expectations
# If using SelectKBest, change 'reduce_dim__n_components' to 'reduce_dim__k'
n_components_options = [10, 20, 30, 40, 50, 60] # Example range for PCA/KBest

param_rfc = {
    'reduce_dim__n_components': n_components_options, # Tune number of dimensions
    'classifier__n_estimators': [50, 100, 200, 300],
    'classifier__max_depth': [None, 5, 10, 15],
    'classifier__min_samples_leaf': [1, 3, 5],
    'classifier__max_features': ['sqrt', 'log2', 0.3] # Important for high dimensions
}

param_svm = {
    'reduce_dim__n_components': n_components_options,
    'classifier__C': [0.1, 1, 10, 50, 100],
    'classifier__kernel': ['linear', 'rbf'],
    # Gamma only relevant for 'rbf' - RandomizedSearch handles this, GridSearch needs care
    'classifier__gamma': ['scale', 'auto', 0.01, 0.1, 1]
}

param_knn = {
    'reduce_dim__n_components': n_components_options,
    'classifier__n_neighbors': np.arange(3, 16, 2), # Odd numbers usually preferred: 3, 5, ..., 15
    'classifier__weights': ['uniform', 'distance'],
    'classifier__metric': ['euclidean', 'manhattan', 'minkowski']
}

# --- 4. Run Experiments ---

model_configs = {
    'RandomForest': (pipe_rfc, param_rfc),
    'SVM': (pipe_svm, param_svm),
    'KNN': (pipe_knn, param_knn)
}

results_summary = [] # To store key results for comparison

for model_name, (pipeline, params) in model_configs.items():
    print(f"===== Running Experiment for: {model_name} =====")
    start_time = time()

    search = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=params,
        n_iter=N_ITER_SEARCH,
        cv=cv_strategy,
        scoring=scoring,
        refit=REFIT_METRIC, # Optimize for this metric
        n_jobs=-1,        # Use all available CPU cores
        random_state=RANDOM_STATE,
        return_train_score=True, # Useful for checking overfitting
        error_score='raise'   # See errors during search
    )

    search.fit(X, y)

    end_time = time()
    print(f"Finished in {end_time - start_time:.2f} seconds.")
    print(f"Best {REFIT_METRIC} score: {search.best_score_:.4f}")
    print(f"Best parameters: {search.best_params_}")

    # Store results for the best model found by the search
    best_model_results = pd.DataFrame(search.cv_results_)
    best_model_row = best_model_results.iloc[search.best_index_]

    result_entry = {
        'Model': model_name,
        'Best Params': search.best_params_,
        f'Mean Valid {REFIT_METRIC}': best_model_row[f'mean_test_{REFIT_METRIC}'],
        f'Std Valid {REFIT_METRIC}': best_model_row[f'std_test_{REFIT_METRIC}'],
        f'Mean Train {REFIT_METRIC}': best_model_row[f'mean_train_{REFIT_METRIC}'],
        f'Std Train {REFIT_METRIC}': best_model_row[f'std_train_{REFIT_METRIC}'],
        'Fit Time (mean, s)': best_model_row['mean_fit_time'],
    }
    # Add other metrics
    for metric in scoring:
        result_entry[f'Mean Valid {metric}'] = best_model_row[f'mean_test_{metric}']
        result_entry[f'Std Valid {metric}'] = best_model_row[f'std_test_{metric}']

    results_summary.append(result_entry)
    print("-" * 30)


# --- 5. Analyze and Compare Results ---
results_df = pd.DataFrame(results_summary).round(4)
results_df = results_df.sort_values(by=f'Mean Valid {REFIT_METRIC}', ascending=False) # Sort by best primary metric

print("\n===== Experiment Results Summary =====")
pd.set_option('display.max_columns', None) # Show all columns
pd.set_option('display.width', 1000)      # Adjust display width
print(results_df)


Data shape: (100, 644), Target shape: (100,), Number of classes: 5
===== Running Experiment for: RandomForest =====
Finished in 72.02 seconds.
Best accuracy score: 0.7600
Best parameters: {'reduce_dim__n_components': 20, 'classifier__n_estimators': 50, 'classifier__min_samples_leaf': 3, 'classifier__max_features': 0.3, 'classifier__max_depth': 10}
------------------------------
===== Running Experiment for: SVM =====
Finished in 9.97 seconds.
Best accuracy score: 0.8500
Best parameters: {'reduce_dim__n_components': 60, 'classifier__kernel': 'linear', 'classifier__gamma': 1, 'classifier__C': 50}
------------------------------
===== Running Experiment for: KNN =====
Finished in 9.71 seconds.
Best accuracy score: 0.6400
Best parameters: {'reduce_dim__n_components': 20, 'classifier__weights': 'distance', 'classifier__n_neighbors': np.int64(5), 'classifier__metric': 'euclidean'}
------------------------------

===== Experiment Results Summary =====
          Model                           